# PartnerCopilot ABSA em português

Extração de aspectos e polaridade em reviews sintéticas de restaurantes com Qwen em 4 bits e um adapter LoRA, com resultados e falhas em texto real.

[Adapter](https://huggingface.co/jnordemann/partnercopilot-absa-qlora-v1) · [Código](https://github.com/JoaoNordemann/partner-copilot) · [Relatório externo](https://github.com/JoaoNordemann/partner-copilot/blob/main/docs/absa-external-evaluation.md) · [Resultados de retrieval](https://github.com/JoaoNordemann/partner-copilot/tree/main/reports/absa-retrieval-v1)

No Kaggle, selecione GPU T4 e habilite internet. O notebook usa a GPU 0 e baixa as revisões públicas sem autenticação.

In [ ]:
import os
import subprocess
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

install = subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transformers==5.20.2", "peft==0.21.2", "bitsandbytes==0.50.2",
    ],
    capture_output=True,
    timeout=300,
)
if install.returncode:
    raise RuntimeError("Falha ao instalar Transformers, PEFT ou bitsandbytes.")

## Carregar o modelo

A base Qwen2.5-1.5B-Instruct usa quantização NF4 com cálculo FP16. O adapter é carregado separadamente. O hash identifica os pesos usados na avaliação.

In [ ]:
import hashlib
import json
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("Selecione uma GPU no Kaggle antes de executar.")
print("GPU", torch.cuda.get_device_name(0))

base_id = "Qwen/Qwen2.5-1.5B-Instruct"
base_revision = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
adapter_id = "jnordemann/partnercopilot-absa-qlora-v1"
adapter_revision = "742d1dcb93d71aee84fda6a84f5ca23dd8c281d6"
adapter = Path(snapshot_download(adapter_id, revision=adapter_revision, token=False))
weights_hash = hashlib.sha256((adapter / "adapter_model.safetensors").read_bytes()).hexdigest()
assert weights_hash == "ef97282c03432c22c9735f419463d1e936b0729ac099052ac96e7b170c7e1d05"

tokenizer = AutoTokenizer.from_pretrained(adapter, local_files_only=True)
base = AutoModelForCausalLM.from_pretrained(
    base_id,
    revision=base_revision,
    token=False,
    dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="sdpa",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    ),
)
model = PeftModel.from_pretrained(base, adapter).eval()

## Extrair aspectos

As seis reviews abaixo foram escritas para a demonstração. A saída aceita tem uma única chave `aspects`, com polaridade `pos` ou `neg`. Aspectos ausentes ficam fora do mapa.

In [ ]:
aspects = {"sabor", "tempo_entrega", "temperatura", "embalagem", "preco", "atendimento"}
instruction = (
    "Classifique os aspectos sabor, tempo_entrega, temperatura, embalagem, preco e atendimento. "
    "Use pos ou neg e omita aspectos sem opinião clara. Responda só com um objeto JSON "
    'com a chave "aspects".\n\nReview:\n'
)
reviews = [
    "A entrega atrasou e a comida chegou fria.",
    "A embalagem veio bem fechada e o sabor estava ótimo.",
    "O atendimento foi educado, mas o preço estava alto.",
    "O pedido chegou antes do prazo, mas a tampa veio solta.",
    "Achei o tempero ruim. Pelo tamanho da porção, o preço valeu a pena.",
    "O pedido chegou quentinho. Mandei uma mensagem e ninguém respondeu.",
]


def unique_keys(pairs):
    value = dict(pairs)
    if len(value) != len(pairs):
        raise ValueError("Chave duplicada")
    return value


def parse_output(raw):
    value = json.loads(raw, object_pairs_hook=unique_keys)
    if not isinstance(value, dict) or set(value) != {"aspects"}:
        raise ValueError("Objeto fora do esquema")
    labels = value["aspects"]
    if not isinstance(labels, dict) or any(
        key not in aspects or polarity not in ("pos", "neg")
        for key, polarity in labels.items()
    ):
        raise ValueError("Aspecto ou polaridade fora da taxonomia")
    return value


def extract(review):
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction + review}],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda:0")
    with torch.inference_mode():
        generated = model.generate(**inputs, do_sample=False, max_new_tokens=96, use_cache=True)
    ids = generated[0, inputs["input_ids"].shape[1] :].tolist()
    eos = model.generation_config.eos_token_id
    eos = {eos} if isinstance(eos, int) else set(eos or ())
    end = next((i for i, token in enumerate(ids) if token in eos), len(ids))
    return tokenizer.decode(ids[:end], skip_special_tokens=False)


valid = 0
for number, review in enumerate(reviews, 1):
    raw = extract(review)
    print(f"{number}. {review}")
    try:
        print(json.dumps(parse_output(raw), ensure_ascii=False))
        valid += 1
    except ValueError:
        print("Saída rejeitada pelo contrato:", raw)
print(f"DEMO_COMPLETA {len(reviews)} reviews | {valid} JSONs válidos")

## Falhas em e-commerce real

A avaliação B2W mediu uma mudança de domínio. A tabela usa resultados agregados, sem redistribuir textos da base.

| Resultado no lote B2W | Contagem |
| --- | ---: |
| Previu sabor ou temperatura | 87/200 |
| Usou chaves fora da taxonomia | 27/200 |
| Erro de sintaxe JSON | 1/200 |
| Válido pelo contrato estrito | 172/200 |
| Sintaticamente válido | 199/200 |

Nos textos com rótulo humano, QLoRA e regras acertaram 6/11 mapas completos. Nas referências vazias, foram 3/19 para o QLoRA e 17/19 para as regras. O desempenho em reviews reais de restaurantes segue sem medição.

A próxima célula ilustra as duas falhas com exemplos inventados.

In [ ]:
examples = [
    ("A luminária iluminou bem a mesa.", '{"aspects":{"sabor":"pos"}}'),
    ("A luminária iluminou bem a mesa.", '{"aspects":{"iluminacao":"pos"}}'),
]
for text, raw in examples:
    print("Ilustração sintética:", text)
    print("Saída ilustrativa:", raw)
    try:
        parse_output(raw)
        print("Contrato válido. Sabor está fora do domínio do texto.")
    except ValueError as error:
        print("Rejeitado pelo contrato.", error)